# Keyloop Scenario A · Reviewer Demo

**Unified Service Scheduler** — Node.js / Hono / SQLite3, with Vue 3 and Tailwind Play CDN (no frontend build).

This notebook executes real commands and HTTP calls against synthetic data. The Node backend is the selected full service layer. Do not mark any test PASS until its output has been inspected.

## M0 / M1 · Bootstrap and safe seed

Repository must already exist at `/kaggle/working/Keyloop-Scenario-A-Unified-Service-Scheduler`. No Docker or Google Drive required. Existing appointment data must not be deleted.

In [ ]:
from pathlib import Path
import subprocess, urllib.request, urllib.error, json, os, time, uuid, datetime
ROOT=Path("/kaggle/working/Keyloop-Scenario-A-Unified-Service-Scheduler")
assert (ROOT/"package-lock.json").exists(), f"Missing source: {ROOT}"
def run(*argv):
    p=subprocess.run(argv,cwd=ROOT,text=True,capture_output=True)
    print("$",*argv,"\n",p.stdout[-2500:],p.stderr[-500:])
    assert p.returncode==0, f"Command failed: {argv}"
run("npm","ci")
run("npm","run","db:migrate")
run("npm","run","db:seed")

## M2 / M3 · Automated verification

Tests cover resource qualification, full-duration intervals, no-overlap triggers, idempotency, HTTP 201/200/409, persistence and a **ready-barrier race across ten independent SQLite processes**.

In [ ]:
p=subprocess.run(["npm","test"],cwd=ROOT,text=True,capture_output=True)
print(p.stdout[-7000:])
assert p.returncode==0, p.stderr[-1000:]

## Launch or reuse Hono on localhost

The notebook does not expose a public service or print any secret token. An optional external tunnel must be configured and authorized separately.

In [ ]:
BASE="http://127.0.0.1:3000"
def api(path,method="GET",payload=None,headers=None):
    req=urllib.request.Request(BASE+path,method=method,data=None if payload is None else json.dumps(payload).encode(),headers={"Content-Type":"application/json",**(headers or {})})
    try:
        with urllib.request.urlopen(req,timeout=12) as r: return r.status,json.load(r)
    except urllib.error.HTTPError as e: return e.code,json.load(e)
server=None
try: print("Existing:",api("/readyz"))
except Exception:
    env={**os.environ,"HOST":"127.0.0.1","PORT":"3000"}
    log=open(ROOT/"evidence"/"notebook-server.log","a")
    server=subprocess.Popen(["node","src/server.js"],cwd=ROOT,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    time.sleep(2)
assert api("/readyz")[0]==200

## Live HTTP proof · available → confirmed → idempotent replay → conflict → read

This creates one *synthetic* persistent booking. To avoid a static historical test date, it chooses the next Monday in the dealership timezone. A new idempotency key targeting the identical unique brake resources must be rejected.

In [ ]:
from urllib.parse import urlencode
today=datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).date()
date=(today+datetime.timedelta(days=(7-today.weekday())%7 or 7)).isoformat()
query=urlencode({"dealershipId":"dlr_001","serviceTypeId":"svc_brake","date":date})
s,found=api("/api/v1/availability?"+query)
assert s==200 and found["data"]["slots"]
body={"customerId":"cus_001","vehicleId":"veh_001","dealershipId":"dlr_001","serviceTypeId":"svc_brake","startAt":found["data"]["slots"][0]["startAt"]}
key="review-"+uuid.uuid4().hex
one,created=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
two,replayed=api("/api/v1/appointments","POST",body,{"Idempotency-Key":key})
three,conflict=api("/api/v1/appointments","POST",body,{"Idempotency-Key":"review-"+uuid.uuid4().hex})
four,persisted=api("/api/v1/appointments/"+created["data"]["id"])
report={"status":[one,two,three,four],"appointmentId":created["data"]["id"],"sameId":created["data"]["id"]==replayed["data"]["id"]}
print(json.dumps(report,indent=2))
assert report["status"]==[201,200,409,200] and report["sameId"]

## Reviewer evidence and frontend entry point

The frontend is served on the exact same port, with browser imports to Vue ESM and Tailwind Play CDN. Browser keyboard, accessibility, responsiveness and styling should be reviewed independently.

In [ ]:
run("npm","run","evidence")
print("GET /:",urllib.request.urlopen(BASE+"/").status)
print("GET /js/main.js:",urllib.request.urlopen(BASE+"/js/main.js").status)
print("Review URL on this machine:",BASE)

## Video narrative (5–10 minutes)

1. Explain the scenario and three Keyloop acceptance criteria.
2. Architecture and technology decisions; note Tailwind Play CDN is demo-only and SQLite is one-host/one-writer.
3. Availability check requires both a qualified technician and a compatible bay across the full half-open interval.
4. Demonstrate actual booking, replay, conflict and persistence responses.
5. Show 10-process race, SQLite triggers, synthetic data and evidence.
6. Discuss AI collaboration: design steering, verification, Sunday fixture correction and UTC-offset contract fix.
7. Wrap up with limitations and lessons learned.